In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors, cm

# ================== CONFIG ==================
PATH = "../../.."
# DATASETS = ["NCT", "PanNuke", "PANDA", "PatchCAM"]  # edit as needed
DATASETS = ["PatchCamelyon"]  # edit as needed
PERTURB_ORDER = None  # or e.g. ["brightness","contrast","gaussian_noise","gaussian_blur","crop","rotation","zoom_out","zoom_in","jpeg"]

# Exclude these models (substring, case-insensitive)
EXCLUDE_MODELS_SUBSTR = ["h0-mini", "virchow", "virchow2"]

# Plot conventions
DROP_ZERO   = {"crop", "jpeg", "zoom_out", "zoom_in"}
INVERT_AXIS = {"crop", "jpeg", "zoom_out"}      # flip x-axis
BASELINE_X  = {"jpeg": 100.0, "zoom_out": 1.0}  # injected display baseline
CROP_ANCHOR_STRENGTH = 225.0

FIGSIZE = (14, 10)
LINEWIDTH = 1.2
MARKER = "o"
MARKERSIZE = 3.5
LEGEND_FONTSIZE = 18
LEGEND_TITLE = "Model"

# ===== Fixed colour mapping to match your legend (tab20 pairs) =====
TAB20 = cm.get_cmap("tab20").colors  # 20 RGBA tuples
MODEL_COLORS = {
    "ResNet18":     mcolors.to_hex(TAB20[0]),   # dark blue
    "ResNet50":     mcolors.to_hex(TAB20[1]),   # light blue
    "UNI":          mcolors.to_hex(TAB20[2]),   # orange
    "UNI2":         mcolors.to_hex(TAB20[3]),   # light orange
    "GigaPath":     mcolors.to_hex(TAB20[4]),   # green
    "Virchow":      mcolors.to_hex(TAB20[5]),   # light green
    "Virchow2":     mcolors.to_hex(TAB20[6]),   # red
    "H0-mini":      mcolors.to_hex(TAB20[7]),   # salmon
    "H-Optimus-0":  mcolors.to_hex(TAB20[8]),   # purple
    "H-Optimus-1":  mcolors.to_hex(TAB20[9]),   # lavender
    "EXAONEPath":   mcolors.to_hex(TAB20[10]),  # brown
    "Hibou-B":      mcolors.to_hex(TAB20[11]),  # light brown
    "Hibou-L":      mcolors.to_hex(TAB20[12]),  # pink
    "Phikon v1":    mcolors.to_hex(TAB20[13]),  # light pink
    "Phikon v2":    mcolors.to_hex(TAB20[14]),  # gray
}
MODEL_ORDER = list(MODEL_COLORS.keys())

def _color_for_model(name: str) -> str:
    """Deterministic fallback colour from tab20 if a model isn't in the fixed map."""
    if name in MODEL_COLORS:
        return MODEL_COLORS[name]
    # stable index from name hash
    idx = abs(hash(name)) % len(TAB20)
    return mcolors.to_hex(TAB20[idx])
# ===================================================================

# ================== HELPERS ==================
def _resolve_dataset_dir_for_read(base_path, dataset_name):
    for c in (os.path.join(base_path, "Stats", dataset_name),
              os.path.join(base_path, "stats", dataset_name)):
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"No Stats/stats dir for dataset '{dataset_name}' under {base_path}")

def _output_dir(base_path, dataset_name):
    out_root = os.path.join(base_path, "Stats", dataset_name, "similarity_aggregate")
    os.makedirs(out_root, exist_ok=True)
    return out_root

def _is_excluded(model_name: str) -> bool:
    ml = model_name.lower()
    return any(sub in ml for sub in EXCLUDE_MODELS_SUBSTR)

def _list_models(ds_dir):
    models = [m for m in os.listdir(ds_dir) if os.path.isdir(os.path.join(ds_dir, m))]
    models = [m for m in models if not _is_excluded(m)]
    # sort to match legend order first, then alphabetically for unknowns
    models.sort(key=lambda m: (MODEL_ORDER.index(m) if m in MODEL_ORDER else 999, m))
    return models

def _collect_perturbations(ds_dir, models):
    perts = set()
    for m in models:
        emb_dir = os.path.join(ds_dir, m, "embeddings")
        if not os.path.isdir(emb_dir):
            continue
        for p in glob.glob(os.path.join(emb_dir, "*_stats.xlsx")):
            perts.add(os.path.basename(p).replace("_stats.xlsx", ""))
    return sorted(perts)

def _load_normalized_curve(stat_path, perturb):
    df = pd.read_excel(stat_path)
    x_all = pd.to_numeric(df.iloc[:, 0], errors="coerce").to_numpy()
    y_all = pd.to_numeric(df.iloc[:, 1], errors="coerce").to_numpy()

    baseline_y = float(y_all[x_all == 0.0][0]) if np.any(x_all == 0.0) else float(y_all[0])

    if perturb in BASELINE_X:
        mask = x_all != 0.0
        x_rest, y_rest = x_all[mask], y_all[mask]
        x = np.concatenate(([BASELINE_X[perturb]], x_rest))
        y = np.concatenate(([baseline_y], y_rest))
    else:
        mask = (x_all != 0.0) if perturb in DROP_ZERO else np.ones_like(x_all, dtype=bool)
        x, y = x_all[mask], y_all[mask]

    if perturb == "crop":
        if x.size:
            idx_anchor = np.where(np.isclose(x, CROP_ANCHOR_STRENGTH, atol=1e-6))[0]
            denom = float(y[idx_anchor[0]]) if idx_anchor.size > 0 else float(y[0])
        else:
            denom = 1.0
        denom = denom if denom != 0 else 1.0
        y_norm = y / denom
    else:
        denom = baseline_y if baseline_y != 0 else 1.0
        y_norm = y / denom

    order = np.argsort(x)
    return x[order], y_norm[order]

def _compute_dataset_y_bounds(base_path, dataset_name, perts_used):
    ds_dir = _resolve_dataset_dir_for_read(base_path, dataset_name)
    models = _list_models(ds_dir)
    if not models or not perts_used:
        return 0.0, 1.1

    vals_min = np.inf
    vals_max = -np.inf
    for model in models:
        emb_dir = os.path.join(ds_dir, model, "embeddings")
        if not os.path.isdir(emb_dir):
            continue
        for perturb in perts_used:
            stat_path = os.path.join(emb_dir, f"{perturb}_stats.xlsx")
            if not os.path.isfile(stat_path):
                continue
            try:
                _, y_norm = _load_normalized_curve(stat_path, perturb)
            except Exception:
                continue
            if y_norm.size:
                vals_min = min(vals_min, float(np.nanmin(y_norm)))
                vals_max = max(vals_max, float(np.nanmax(y_norm)))
    if not np.isfinite(vals_min) or not np.isfinite(vals_max):
        return 0.0, 1.1
    y_min = max(0.0, 0.9 * vals_min)
    y_max = 1.05 * vals_max
    if y_max <= y_min:
        y_max = y_min + 0.1
    return y_min, y_max
# ===========================================

def _plot_dataset_grid(base_path, dataset_name, perturb_order=None):
    """
    Creates one 3x3 grid per dataset with a single legend on the right.
    Saves to: Stats/{dataset}/similarity_aggregate/{dataset}_similarity_grid.png
    """
    ds_dir = _resolve_dataset_dir_for_read(base_path, dataset_name)
    out_dir = _output_dir(base_path, dataset_name)

    models = _list_models(ds_dir)
    if not models:
        print(f"[WARN] No models for dataset '{dataset_name}'. Skipping.")
        return

    perts_all = _collect_perturbations(ds_dir, models)
    if not perts_all:
        print(f"[WARN] No perturbations found for '{dataset_name}'. Skipping.")
        return

    if perturb_order:
        perts_used = [p for p in perturb_order if p in perts_all][:9]
    else:
        perts_used = perts_all[:9]

    y_min, y_max = _compute_dataset_y_bounds(base_path, dataset_name, perts_used)

    fig, axes = plt.subplots(3, 3, figsize=FIGSIZE, constrained_layout=True)
    axes = axes.flatten()

    legend_handles = {}
    for i, perturb in enumerate(perts_used):
        ax = axes[i]
        plotted_any = False

        for model in models:
            emb_dir = os.path.join(ds_dir, model, "embeddings")
            stat_path = os.path.join(emb_dir, f"{perturb}_stats.xlsx")
            if not os.path.isfile(stat_path):
                continue
            try:
                x, y_norm = _load_normalized_curve(stat_path, perturb)
            except Exception as e:
                print(f"[SKIP] {dataset_name}/{model}/{perturb}: {e}")
                continue

            if x.size == 0 or y_norm.size == 0:
                continue

            color = _color_for_model(model)
            line, = ax.plot(
                x, y_norm,
                marker=MARKER, markersize=MARKERSIZE,
                linewidth=LINEWIDTH, color=color, label=model
            )
            legend_handles[model] = line
            plotted_any = True

        ax.set_title(perturb)
        ax.set_xlabel("strength")
        ax.set_ylabel("normalized similarity")
        ax.grid(True, alpha=0.3)
        ax.set_ylim(y_min, y_max)
        ax.margins(x=0.01, y=0.02)

        if perturb in INVERT_AXIS:
            ax.invert_xaxis()

        if not plotted_any:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])

    # Hide any unused panels if <9 perts found
    for j in range(len(perts_used), 9):
        axes[j].axis("off")

    # Single legend on the right, minimal dead space; order by MODEL_ORDER then alpha
    present_models = sorted(legend_handles.keys(),
                            key=lambda m: (MODEL_ORDER.index(m) if m in MODEL_ORDER else 999, m))
    handles = [legend_handles[m] for m in present_models]
    labels  = present_models
    n_models = len(labels)
    ncol = 1 if n_models <= 12 else (2 if n_models <= 24 else 3)
    fig.legend(handles, labels, title=LEGEND_TITLE,
               loc="center left", bbox_to_anchor=(1.01, 0.5),
               frameon=False, fontsize=LEGEND_FONTSIZE, title_fontsize=LEGEND_FONTSIZE,
               ncol=ncol, handlelength=1.5, handletextpad=0.6)

    out_png = os.path.join(out_dir, f"{dataset_name}_similarity_grid.png")
    fig.savefig(out_png, dpi=220, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f"[OK] Saved grid → {out_png}")

# ----------------- RUN -------------------
for ds in DATASETS:
    try:
        _plot_dataset_grid(
            base_path=PATH,
            dataset_name=ds,
            perturb_order=PERTURB_ORDER
        )
    except FileNotFoundError as e:
        print(f"[SKIP] {ds}: {e}")

C:\Users\Dhyey\AppData\Local\Temp\ipykernel_33532\299093221.py:31: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  TAB20 = cm.get_cmap("tab20").colors  # 20 RGBA tuples


[OK] Saved grid → Stats\PatchCamelyon\similarity_aggregate\PatchCamelyon_similarity_grid.png
